**Project Description:**

This time series data project uses a Beijing Air Quality dataset ([Beijing Multi-Site Air Quality](https://https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data$0)).
It contains data from 12 different stations. For this project we will be using the data from the Wanshouxigong station.

The data will be used to predict PM2.5 concentration using PM10, SO2, NO2, CO, O3, temperature, precipitation, DEWP, rain, wd, and WSPM as features.

Our motivation to predict PM2.5 is that this particulate matter creates a health risk. If we are able to accurately predict high levels of PM2.5 in the air, people in the local area can prepare and reschedule acitivities that may expose them to the harmful air in advance.

**Stage 1: Data, baselines, MLP**

In [63]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import tensorflow as tf
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten


In [30]:
df = pd.read_csv("/content/PRSA_Data_Wanshouxigong_20130301-20170228.csv")
df['Date'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']]) # Creates a date column out of day, month, year, hour columns
df = df.sort_values("Date").set_index("Date").asfreq("H") # Makes the date column the index and adds missing rows in timeline as blank.

# Data Exploration
display(df.head(10))
print("Date range:", df.index.min().date(), "to", df.index.max().date())
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))



/tmp/ipykernel_1269/1256170079.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.sort_values("Date").set_index("Date").asfreq("H")


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
Date,,,,,,,,,,,,,,,,,,
2013-03-01 00:00:00,1,2013,3,1,0,9.0,9.0,6.0,17.0,200.0,62.0,0.3,1021.9,-19.0,0.0,WNW,2.0,Wanshouxigong
2013-03-01 01:00:00,2,2013,3,1,1,11.0,11.0,7.0,14.0,200.0,66.0,-0.1,1022.4,-19.3,0.0,WNW,4.4,Wanshouxigong
2013-03-01 02:00:00,3,2013,3,1,2,8.0,8.0,NaN,16.0,200.0,59.0,-0.6,1022.6,-19.7,0.0,WNW,4.7,Wanshouxigong
2013-03-01 03:00:00,4,2013,3,1,3,8.0,8.0,3.0,16.0,NaN,NaN,-0.7,1023.5,-20.9,0.0,NW,2.6,Wanshouxigong
2013-03-01 04:00:00,5,2013,3,1,4,8.0,8.0,3.0,NaN,300.0,36.0,-0.9,1024.1,-21.7,0.0,WNW,2.5,Wanshouxigong
2013-03-01 05:00:00,6,2013,3,1,5,10.0,10.0,4.0,8.0,200.0,64.0,-1.6,1024.7,-21.1,0.0,NE,2.0,Wanshouxigong
2013-03-01 06:00:00,7,2013,3,1,6,8.0,8.0,6.0,13.0,300.0,61.0,-2.4,1025.4,-20.3,0.0,NE,2.3,Wanshouxigong
2013-03-01 07:00:00,8,2013,3,1,7,8.0,8.0,8.0,20.0,300.0,54.0,-0.8,1026.7,-19.9,0.0,NNE,2.0,Wanshouxigong
2013-03-01 08:00:00,9,2013,3,1,8,3.0,6.0,9.0,23.0,300.0,50.0,0.4,1027.3,-19.4,0.0,NE,2.7,Wanshouxigong


Date range: 2013-03-01 to 2017-02-28
Shape: (35064, 18)

Columns: ['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']


In [67]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Identifying features and target
numeric_variables = ['PM2.5','PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
cat_variables = ['wd']
target = 'PM2.5'

# Train/Validation Split
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

train = df.iloc[:train_end].copy()
val   = df.iloc[train_end:val_end].copy()
test  = df.iloc[val_end:].copy()

# Imputing data using forward fill
train[numeric_variables] = train[numeric_variables].ffill()
val[numeric_variables] = val[numeric_variables].ffill()
test[numeric_variables] = test[numeric_variables].ffill()

train[cat_variables] = train[cat_variables].ffill()
val[cat_variables] = val[cat_variables].ffill()
test[cat_variables] = test[cat_variables].ffill()

# Scaling data (numeric variables only)
scaler = StandardScaler()
train_num_scaled = scaler.fit_transform(train[numeric_variables].to_numpy())
val_num_scaled = scaler.transform(val[numeric_variables].to_numpy())
test_num_scaled = scaler.transform(test[numeric_variables].to_numpy())

# One hot encoding the wind direction
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_cat_encoded = encoder.fit_transform(train[cat_variables])
val_cat_encoded = encoder.transform(val[cat_variables])
test_cat_encoded = encoder.transform(test[cat_variables])

# Combine the processed categorical and numeric arrays
train_combined = np.hstack((train_num_scaled, train_cat_encoded))
val_combined = np.hstack((val_num_scaled, val_cat_encoded))
test_combined = np.hstack((test_num_scaled, test_cat_encoded))

# Make the windows for training
LOOKBACK = 24 # Lookback is 24 for 24 hours
def make_windows(values, lookback):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i-lookback:i])
        y.append(values[i])
    return np.asarray(X), np.asarray(y)

X_train, y_train = make_windows(train_combined, LOOKBACK)
X_val, y_val = make_windows(val_combined, LOOKBACK)
X_test, y_test = make_windows(test_combined, LOOKBACK)




In [68]:
# Non neural network baseline
pm25_idx = numeric_variables.index('PM2.5') # Looks at what column PM2.5 is in dataset

# Take the average of all the PM2.5 values in the window
y_pred_baseline = np.mean(X_val[:, :, pm25_idx], axis=1)

# Compare the average of past values to actual future target
baseline_mae = mean_absolute_error(y_val[:, 0], y_pred_baseline)

print(f"Scaled Moving Average MAE: {baseline_mae:.4f}")

Scaled Moving Average MAE: 0.4458


In [ ]:
# MLP

# Extract ONLY the PM2.5 column for the neural networks to predict
y_train_pm25 = y_train[:, pm25_idx]
y_val_pm25 = y_val[:, pm25_idx]
y_test_pm25 = y_test[:, pm25_idx]


num_features = X_train.shape[-1]

mlp_model = Sequential([
    Input(shape=(LOOKBACK, num_features)),
    Flatten(),
    Dense(32, activation="relu"),
    Dense(1)
])

mlp_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

mlp_history = mlp_model.fit(
    X_train, y_train_pm25,
    validation_data=(X_val, y_val_pm25),
    epochs=30,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

mlp_predictions = mlp_model.predict(X_val)
mlp_mae = mean_absolute_error(y_val_pm25, mlp_predictions)

# CNN
tf.keras.utils.set_random_seed(SEED)

cnn_model = Sequential([
    Input(shape=(LOOKBACK, num_features)),
    Conv1D(filters=16, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(16, activation="relu"),
    Dense(1)
])

cnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

cnn_history = cnn_model.fit(
    X_train, y_train_pm25,
    validation_data=(X_val, y_val_pm25),
    epochs=60,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

cnn_predictions = cnn_model.predict(X_val)
cnn_mae = mean_absolute_error(y_val_pm25, cnn_predictions)



print(f"Baseline (Moving Average): {baseline_mae:.4f}")
print(f"MLP Model: {mlp_mae:.4f}")
print(f"CNN Model: {cnn_mae:.4f}")

Epoch 1/30
767/767 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.2786 - val_loss: 0.2535
Epoch 2/30
767/767 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1360 - val_loss: 0.2066
Epoch 3/30
767/767 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.1700 - val_loss: 0.1586
Epoch 4/30
767/767 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.1284 - val_loss: 0.1251
Epoch 5/30
767/767 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.1034 - val_loss: 0.1121
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Epoch 1/60
112/767 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.1002

In [ ]:
# Comparision table

In [ ]:
# Conclusion